# LLM Classification Finetuning — TF-IDF + LightGBM baseline
Self-contained submission notebook. Reads the competition data, trains a 3-class
LightGBM on handcrafted + TF-IDF features, and writes `submission.csv`.
Runs on CPU, no internet required.

In [ ]:
import json, re, time, glob, os
import numpy as np, pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
import lightgbm as lgb

# Auto-detect the competition data folder (robust to input path naming).
print('contents of /kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'MISSING')
cands = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert cands, 'train.csv not found under /kaggle/input - attach the competition data (Add Input).'
INP = os.path.dirname(cands[0])
print('using INP =', INP)
OUT = '/kaggle/working/submission.csv'
TARGETS = ['winner_model_a', 'winner_model_b', 'winner_tie']
SEED = 42

In [ ]:
# --- text parsing: prompt/response columns are JSON-encoded lists of turns ---
def parse_list(x):
    if isinstance(x, list):
        return [str(t) for t in x]
    if not isinstance(x, str):
        return ['']
    try:
        v = json.loads(x)
    except Exception:
        return [x]
    if isinstance(v, list):
        return ['' if t is None else str(t) for t in v]
    return ['' if v is None else str(v)]

def load(name):
    df = pd.read_csv(f'{INP}/{name}')
    for c in ['prompt', 'response_a', 'response_b']:
        if c in df.columns:
            df[c + '_text'] = df[c].map(parse_list).map(lambda ts: '\n'.join(ts))
    return df

train, test = load('train.csv'), load('test.csv')
y = np.argmax(train[TARGETS].values, axis=1)
print('train', train.shape, 'test', test.shape, 'class balance', np.bincount(y) / len(y))

In [ ]:
# --- handcrafted numeric features (length / formatting / A-B differences) ---
_CODE, _LIST, _HDR = re.compile(r'```'), re.compile(r'(?m)^\s*(?:[-*+]|\d+\.)\s'), re.compile(r'(?m)^#{1,6}\s')
def stats(t):
    w = t.split(); nw = len(w)
    return {'n_char': len(t), 'n_word': nw, 'n_line': t.count('\n') + 1,
            'n_code': len(_CODE.findall(t)) // 2, 'n_list': len(_LIST.findall(t)),
            'n_hdr': len(_HDR.findall(t)), 'awl': (sum(len(x) for x in w) / nw) if nw else 0.0}

def num_feats(df):
    rows = []
    for p, a, b in zip(df['prompt_text'], df['response_a_text'], df['response_b_text']):
        pa, pb, pp = stats(a), stats(b), stats(p)
        f = {f'prompt_{k}': v for k, v in pp.items()}
        for k in pa:
            f[f'a_{k}'], f[f'b_{k}'] = pa[k], pb[k]
            f[f'diff_{k}'] = pa[k] - pb[k]
            d = pa[k] + pb[k]; f[f'ratio_{k}'] = (pa[k] - pb[k]) / d if d else 0.0
        f['a_empty'], f['b_empty'] = float(not a.strip()), float(not b.strip())
        rows.append(f)
    return pd.DataFrame(rows, index=df.index).astype(np.float32).values

In [ ]:
# --- build feature matrix: numeric + TF-IDF(prompt+A), TF-IDF(prompt+B), and their diff ---
t0 = time.time()
vec = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
tr_a = (train['prompt_text'] + ' ' + train['response_a_text']).tolist()
tr_b = (train['prompt_text'] + ' ' + train['response_b_text']).tolist()
vec.fit(tr_a + tr_b)

def matrix(df):
    a = vec.transform((df['prompt_text'] + ' ' + df['response_a_text']).tolist())
    b = vec.transform((df['prompt_text'] + ' ' + df['response_b_text']).tolist())
    return sp.hstack([sp.csr_matrix(num_feats(df)), a, b, a - b], format='csr')

Xtr, Xte = matrix(train), matrix(test)
print('Xtr', Xtr.shape, 'Xte', Xte.shape, f'({time.time() - t0:.0f}s)')

In [ ]:
# --- train with a validation holdout to pick #rounds, then predict test ---
params = dict(objective='multiclass', num_class=3, learning_rate=0.05, num_leaves=63,
              feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1,
              min_child_samples=50, seed=SEED, verbose=-1)
xt, xv, yt, yv = train_test_split(Xtr, y, test_size=0.1, stratify=y, random_state=SEED)
m = lgb.train(params, lgb.Dataset(xt, yt), num_boost_round=3000,
              valid_sets=[lgb.Dataset(xv, yv)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
print(f'val log_loss = {log_loss(yv, m.predict(xv), labels=[0,1,2]):.4f}  best_iter = {m.best_iteration}')

In [ ]:
# --- refit on ALL train at the chosen round count, predict, write submission ---
final_rounds = int(m.best_iteration / 0.9) + 1
full = lgb.train(params, lgb.Dataset(Xtr, y), num_boost_round=final_rounds)
sub = test[['id']].copy()
sub[TARGETS] = full.predict(Xte)
sub.to_csv(OUT, index=False)
print('wrote', OUT, sub.shape)
sub.head()